# Example 01 · LLM with Tools (ReAct Agent)

**Source:** [LangGraph Tutorial — Build a ReAct Agent](https://langchain-ai.github.io/langgraph/tutorials/introduction/)

---

## Objectives

By the end of this notebook you will be able to:

1. Explain the **ReAct (Reason + Act)** loop and why it works
2. Define tools using the `@tool` decorator with type annotations
3. Build a `StateGraph` with `llm_call` and `tool_node` nodes
4. Implement conditional routing to decide when to call tools vs. stop
5. Run the agent end-to-end and read the full message trace

---

## Background

### What is a ReAct Agent?

A **ReAct agent** interleaves *reasoning* (asking the LLM what to do next) with
*acting* (calling a tool and observing the result). This loop continues until the
LLM decides it has enough information to answer without calling any more tools.

```
START → llm_call ──(has tool calls?)──► tool_node ──► llm_call
                 └──(no tool calls)──► END
```

### LangGraph StateGraph

The agent is a **directed graph** with two nodes:

| Node | Role |
|------|------|
| `llm_call` | Sends the full message history to the LLM; appends its response |
| `tool_node` | Executes every pending tool call; appends the results |

**State accumulation:** `AgentState.messages` uses `operator.add` as a reducer —
each node *appends* rather than replaces. The LLM always sees the full history.

Run cells **top to bottom**.

## Setup

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so the 'common' package is importable
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import operator
from typing import Literal
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import AnyMessage, SystemMessage, ToolMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END

from common.env import get_env  # noqa: F401 — triggers .env loading
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

print("✓ All imports loaded")

✓ All imports loaded


## Step 1 · Define Tools

Tools are plain Python functions decorated with `@tool`.
LangChain reads the function's type annotations and docstring to generate the
JSON schema the LLM uses when deciding to call a tool.

In [2]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a by b."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add a and b."""
    return a + b

@tool
def divide(a: int, b: float) -> float:
    """Divide a by b."""
    return a / b

# Name → tool mapping used by tool_node to dispatch calls
_tools = [add, multiply, divide]
_tools_by_name = {t.name: t for t in _tools}

print(f"✓ {len(_tools)} tools defined: {[t.name for t in _tools]}")

✓ 3 tools defined: ['add', 'multiply', 'divide']


## Step 2 · Build the Agent Graph

Four parts assembled in a single cell for clarity:
1. **`AgentState`** — TypedDict holding the full message history
2. **Node functions** — `llm_call` and `tool_node`
3. **Routing function** — `should_continue` decides the next node
4. **`StateGraph`** — wires everything together and compiles the runnable

In [3]:
# ── State ─────────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    # operator.add means each node's output is appended, not replaced
    messages: Annotated[list[AnyMessage], operator.add]

# ── Model bound to tools (done once to avoid re-initialising on every call) ──
_model = create_llm().bind_tools(_tools)

_SYSTEM_PROMPT = (
    "You are a math assistant that uses tools for arithmetic. "
    "Call one tool at a time and wait for the result before deciding the next step."
)

# ── Nodes ─────────────────────────────────────────────────────────────────────
def llm_call(state: AgentState) -> dict:
    """Invoke the LLM with the full message history."""
    response = _model.invoke([SystemMessage(content=_SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState) -> dict:
    """Execute every tool call in the latest AI message."""
    results = []
    for tc in state["messages"][-1].tool_calls:
        observation = _tools_by_name[tc["name"]].invoke(tc["args"])
        # Attach name so the LLM can match results to calls during parallel execution
        results.append(ToolMessage(content=str(observation), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}

# ── Routing ───────────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tool_node", "__end__"]:
    """If the latest message has pending tool calls, run them. Otherwise end."""
    return "tool_node" if state["messages"][-1].tool_calls else END

# ── Graph ─────────────────────────────────────────────────────────────────────
def build_agent():
    g = StateGraph(AgentState)
    g.add_node("llm_call", llm_call)
    g.add_node("tool_node", tool_node)
    g.add_edge(START, "llm_call")
    g.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    g.add_edge("tool_node", "llm_call")
    return g.compile()

print("✓ Agent graph defined — ready to compile")

✓ Agent graph defined — ready to compile


## Step 3 · Run the Agent

The agent will reason through the problem step by step:
1. LLM → call `multiply(12345, 17)` → result `209865`
2. LLM → call `divide(209865, 3)` → result `69955.0`
3. LLM → no more tool calls → returns final answer

Each `.pretty_print()` below shows one message in the chain.

In [4]:
agent = build_agent()
question = "What is 12345 multiplied by 17, then divided by 3?"

print(f"Question: {question}")
print("=" * 60)

handler = create_langfuse_handler()
config = build_langfuse_config(
    handler,
    session_id="s_01_notebook",
    user_id="demo-user",
    trace_name="Notebook: Math Question",
)
config["configurable"] = {"thread_id": "nb-s01"}

result = agent.invoke({"messages": [HumanMessage(content=question)]}, config=config)

for msg in result["messages"]:
    msg.pretty_print()

print(f"\nTrace: {get_langfuse_host()}")

Question: What is 12345 multiplied by 17, then divided by 3?
================================ Human Message =================================

What is 12345 multiplied by 17, then divided by 3?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_lrkHVegZTwdghBLiHTdtgeR9)
 Call ID: call_lrkHVegZTwdghBLiHTdtgeR9
  Args:
    a: 12345
    b: 17
================================= Tool Message =================================
Name: multiply

209865
================================== Ai Message ==================================
Tool Calls:
  divide (call_SZng5mnXWbCMy87dO1XLdvl3)
 Call ID: call_SZng5mnXWbCMy87dO1XLdvl3
  Args:
    a: 209865
    b: 3
================================= Tool Message =================================
Name: divide

69955.0
================================== Ai Message ==================================

The result of \( 12345 \times 17 \div 3 \) is \( 69955.0 \).

Trace: http://localhost:3000


---

## Summary

| Concept | API | Purpose |
|---------|-----|---------|
| Tool definition | `@tool` decorator | Generates JSON schema from type hints + docstring |
| Agent state | `AgentState` + `operator.add` | Accumulates the full message history |
| LLM node | `llm_call` | Calls the model; returns its response |
| Tool node | `tool_node` | Executes pending tool calls; returns results |
| Routing | `should_continue` + `add_conditional_edges` | Decides loop vs. stop |
| Graph compilation | `StateGraph.compile()` | Produces a runnable agent |

### Key Takeaways

1. **Tools are just Python functions** — `@tool` does the schema generation for you
2. **State is append-only** — `operator.add` means no message is ever lost
3. **The routing function is the brain** — one conditional edge controls the entire loop
4. **The graph is composable** — swap the LLM, add nodes, or change routing without touching other parts
